In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
from sodapy import Socrata
from datetime import date
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

BASE_DIR     = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
HIST_PATH    = os.path.join("C:/Users/Usuario/OneDrive - Global Green Growth Institute/Documentos/2025/Outputs/Output4/Indicadores/precipitacion diaria/")

os.chdir(BASE_DIR)
print("Base dir :", BASE_DIR)
print("Histórico:", HIST_PATH)


Base dir : C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data
Histórico: C:/Users/Usuario/OneDrive - Global Green Growth Institute/Documentos/2025/Outputs/Output4/Indicadores/precipitacion diaria/


## Paso 1 – Descarga datos del día más reciente (API IDEAM)

In [2]:
DATASET_ID = "s54a-sgyg"
client = Socrata("www.datos.gov.co", None)

# Obtener fecha más reciente con ORDER BY en lugar de max() — más confiable
latest = client.get(DATASET_ID, select="fechaobservacion", order="fechaobservacion DESC", limit=1)
fecha_max_api = latest[0]["fechaobservacion"][:10]
print(f"Última fecha disponible en API: {fecha_max_api}")

# Última fecha ya guardada en el histórico
HIST_PATH_CHECK = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\precipitacion diaria\precipitacion_diaria.xlsx"
import os
if os.path.exists(HIST_PATH_CHECK):
    df_check = pd.read_excel(HIST_PATH_CHECK, usecols=["fecha"])
    fecha_min_descarga = pd.to_datetime(df_check["fecha"]).max().strftime("%Y-%m-%d")
    print(f"Última fecha en histórico    : {fecha_min_descarga}")
else:
    fecha_min_descarga = fecha_max_api  # primera vez

# Fechas pendientes de descargar (entre histórico+1 y API max)
fechas_pendientes = pd.date_range(
    start=pd.to_datetime(fecha_min_descarga) + pd.Timedelta(days=1),
    end=pd.to_datetime(fecha_max_api),
    freq="D"
).strftime("%Y-%m-%d").tolist()

if not fechas_pendientes:
    print("El histórico ya está al día. Nada nuevo que descargar.")
else:
    print(f"\nFechas pendientes ({len(fechas_pendientes)}): {fechas_pendientes[0]} → {fechas_pendientes[-1]}")

Última fecha disponible en API: 2026-04-22


Última fecha en histórico    : 2026-04-21

Fechas pendientes (1): 2026-04-22 → 2026-04-22


## Paso 2 – Agregar lecturas 10-min a nivel diario por estación

In [3]:
all_dias = []

for fecha_mapa in fechas_pendientes if fechas_pendientes else [fecha_max_api]:
    print(f"Descargando {fecha_mapa} ...", end=" ")
    where = (
        f"fechaobservacion >= '{fecha_mapa}T00:00:00' "
        f"AND fechaobservacion <= '{fecha_mapa}T23:59:59'"
    )
    records, offset = [], 0
    while True:
        batch = client.get(DATASET_ID, where=where, limit=100_000, offset=offset)
        if not batch:
            break
        records.extend(batch)
        offset += 100_000

    if not records:
        print("sin datos.")
        continue

    df_api = pd.DataFrame.from_records(records)
    for col in ("valorobservado", "latitud", "longitud"):
        df_api[col] = pd.to_numeric(df_api[col], errors="coerce")
    df_api = df_api.dropna(subset=["latitud", "longitud", "valorobservado"])
    df_api = df_api[df_api["valorobservado"] >= 0]

    STATION_COLS_API = [
        "codigoestacion", "nombreestacion", "departamento",
        "municipio", "zonahidrografica", "latitud", "longitud",
    ]
    df_dia = (
        df_api.groupby(STATION_COLS_API, as_index=False)
        .agg(
            precip_acum_diaria = ("valorobservado", "sum"),
            precip_max_10min   = ("valorobservado", "max"),
            precip_min_10min   = ("valorobservado", "min"),
            precip_media_10min = ("valorobservado", "mean"),
            n_lecturas         = ("valorobservado", "count"),
        )
    )
    df_dia["fecha"] = pd.to_datetime(fecha_mapa)
    all_dias.append(df_dia)
    print(f"{len(df_dia):,} estaciones.")

client.close()

if all_dias:
    df_nuevo_total = pd.concat(all_dias, ignore_index=True)
    print(f"\nTotal registros nuevos: {len(df_nuevo_total):,} ({len(all_dias)} día(s))")
else:
    df_nuevo_total = pd.DataFrame()
    print("No hay datos nuevos.")

Descargando 2026-04-22 ... 

514 estaciones.

Total registros nuevos: 514 (1 día(s))


In [4]:
#df_dia.to_excel(HIST_PATH, index=False)

## Paso 3 – Concatenar con histórico y guardar

In [5]:
HIST_PATH = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\precipitacion diaria\precipitacion_diaria.xlsx"

COLS = [
    "codigoestacion", "nombreestacion", "departamento", "municipio",
    "zonahidrografica", "latitud", "longitud", "fecha",
    "precip_acum_diaria", "precip_max_10min", "precip_min_10min",
    "precip_media_10min", "n_lecturas",
]

def normalizar(df):
    df = df.copy()
    df.columns = df.columns.str.lower()
    for col in COLS:
        if col not in df.columns:
            df[col] = np.nan
    return df[COLS]

if df_nuevo_total.empty:
    print("Nada que guardar.")
else:
    df_hist = normalizar(pd.read_excel(HIST_PATH)) if os.path.exists(HIST_PATH) else pd.DataFrame(columns=COLS)
    df_nuevo_norm = normalizar(df_nuevo_total)

    df_daily = (
        pd.concat([df_hist, df_nuevo_norm], ignore_index=True)
        .drop_duplicates(subset=["codigoestacion", "fecha"], keep="last")
        .sort_values(["codigoestacion", "fecha"])
        .reset_index(drop=True)
    )

    df_daily.to_excel(HIST_PATH, index=False)

    print(f"Guardado: {HIST_PATH}")
    print(f"Registros anteriores : {len(df_hist):,}")
    print(f"Registros nuevos     : {len(df_nuevo_norm):,}")
    print(f"Total                : {len(df_daily):,}")
    print(f"Rango fechas         : {df_daily['fecha'].min().date()} → {df_daily['fecha'].max().date()}")

Guardado: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\precipitacion diaria\precipitacion_diaria.xlsx
Registros anteriores : 15,781
Registros nuevos     : 514
Total                : 16,295
Rango fechas         : 2026-03-18 → 2026-04-22
